## 0 · Vertrekpunt: is de lijst ook goed?

Jullie recommender geeft voor elke lezer een lijst. Dat hij een lijst geeft, zegt nog niet dat het een goede lijst is. Vandaag maak je dat meetbaar: per testlezer ligt vast welke titels je verwacht, en je telt hoeveel daarvan in de aanbevelingen staan. Die cijfers leg je naast een simpele ondergrens, zodat je in het keuzelog kunt laten zien wat jullie recommender toevoegt.

De functies in dit notebook vergelijken twee lijsten met cataloguscodes. Ze weten niets van `X`, vectoren of profielen. Daardoor kun je ze bouwen en testen zonder jullie recommender; die koppel je pas aan het eind van deel 3.

| Deel | Demo | Opdracht |
|---|---|---|
| 0 | Vertrekpunt: de catalogus | – |
| 1 | Zeven vaste testlezers met een verwachting; jullie voegen er drie toe | – |
| 2 | Eén lijst meten: precision, recall en hit | Opdracht 1: `recall_at_k` en `hit_at_k` |
| 3 | Alle testlezers, en een ondergrens | Opdracht 2: `evaluate`, kies een ondergrens en meet jullie eigen recommender |

In [ ]:
import random
import pandas as pd

CATALOG_PATH = "datasets/processed/catalogus_opgeschoond.csv"     # pas aan naar jullie eigen bestand

catalog = pd.read_csv(CATALOG_PATH, sep=";").set_index("cataloguscode")
print(len(catalog), "titels in de catalogus")
catalog[["Titel", "Type materiaal", "Niveau", "genre", "trefwoorden"]].head()

## 1 · Zeven testlezers met een verwachting

Om te meten heb je iets nodig om tegen te meten. Echte leesgegevens zijn er niet, dus de verwachting is met de hand gemaakt. Een *testlezer* is dezelfde dict als in de vorige les, met één veld erbij: in `relevant` staan de titels die deze lezer zou moeten krijgen.

- **De sleutel is `cataloguscode`, niet de titel.** Sommige titels staan meer dan eens in de catalogus; de code is uniek.
- **De harde filters tellen mee.** In `relevant` staan alleen titels die de lezer mag krijgen: niet boven zijn `level`, en geen type uit `avoid`. Een titel die inhoudelijk past maar te moeilijk is, is voor deze lezer een misser.

De zeven testlezers hieronder liggen vast: iedereen meet op dezelfde lezers en dezelfde verwachtingen, dus pas ze niet aan. De verwachtingen zijn opgesteld door alle omschrijvingen te lezen, zonder een recommender te draaien.

| Lezer | Wat wil hij of zij | Waar let je op |
|---|---|---|
| Daan | thrillers en detectives, 2F | brede smaak, veel relevante titels |
| Thomas | de Tweede Wereldoorlog, 3F | het trefwoord is "WOII" of "oorlog"; de tekst zegt "Tweede Wereldoorlog" |
| Fatima | migratie en opnieuw beginnen, 2F | een titel die precies past is 3F en mag dus niet |
| Milan | sport én AI, 3F | twee smaken; AI staat alleen in artikelen, en die hebben geen trefwoorden |
| Lotte | rouw en verlies, 3F | vulde alleen `free_text` in; de catalogus zegt vaker "verlies" dan "rouw" |
| Youssef | recht en criminaliteit, 3F | wil boeken: `avoid` sluit artikelen en blogposts uit |
| Emma | gedichten over identiteit en opgroeien, 2F | smalle interesse; één relevante bundel heeft geen genre |

**Voeg zelf drie testlezers toe** in `own_readers`, onderaan de cel hieronder. Kies lezers die iets testen wat de zeven nog niet testen, bijvoorbeeld een lezer op niveau 3F+, een lezer die alleen `genres` invulde of een lezer die kranten en tijdschriften wil. Vul `relevant` in vóórdat je de recommender voor die lezer draait, en zoek daarbij ook in de omschrijvingen en niet alleen op trefwoord.

In [ ]:
def codes(*numbers):
    """Catalogue codes from their numbers: codes(1, 17) -> ["CAT-00001", "CAT-00017"]."""
    return [f"CAT-{n:05d}" for n in numbers]

daan = {
    "name": "Daan",
    "tags": ["thriller", "mysterie", "geheimen"],
    "genres": ["fictie"],
    "free_text": "een spannende thriller of detective met geheimen en een onverwachte ontknoping",
    "level": "2F",
    "avoid": [],
    "relevant": codes(54, 57, 66, 87, 89, 94, 98, 101, 102, 103, 108, 109, 110, 114, 121, 123, 124, 125,
                      133, 134, 135, 191, 202, 207, 240, 247, 253, 266, 277),
}

thomas = {
    "name": "Thomas",
    "tags": ["oorlog", "WOII", "onderduik"],
    "genres": ["fictie"],
    "free_text": "waargebeurde verhalen uit de Tweede Wereldoorlog, over onderduiken en overleven",
    "level": "3F",
    "avoid": [],
    "relevant": codes(1, 10, 11, 17, 32, 33, 42, 46, 85, 248),
}

fatima = {
    "name": "Fatima",
    "tags": ["migratie", "hoop"],
    "genres": ["fictie"],
    "free_text": "verhalen over mensen die naar een nieuw land komen en daar opnieuw moeten beginnen",
    "level": "2F",
    "avoid": [],
    "relevant": codes(22, 43, 77, 188, 206, 217, 222, 223, 271, 277),
}

milan = {
    "name": "Milan",
    "tags": ["sport"],
    "genres": [],
    "free_text": "boeken over sport, en ik wil ook weten wat AI voor mij gaat betekenen",
    "level": "3F",
    "avoid": [],
    "relevant": codes(48, 106, 111, 115, 184, 240, 268, 272,                    # sport
                      139, 140, 148, 152, 153, 175, 178, 183, 209, 280),        # AI
}

lotte = {
    "name": "Lotte",
    "tags": [],
    "genres": [],
    "free_text": "ik zoek iets over rouw: iemand die een ouder of een vriend verliest en daarmee moet leren leven",
    "level": "3F",
    "avoid": [],
    "relevant": codes(21, 79, 117, 136, 198, 201, 205, 213, 219, 234, 276),
}

youssef = {
    "name": "Youssef",
    "tags": ["criminaliteit", "recht"],
    "genres": [],
    "free_text": "hoe werkt de rechtspraak, en wat gebeurt er met jongeren die de fout in gaan",
    "level": "3F",
    "avoid": ["Online artikel", "Blogpost"],
    "relevant": codes(50, 64, 69, 71, 73, 74, 76, 78, 87, 91, 93),
}

emma = {
    "name": "Emma",
    "tags": [],
    "genres": ["poëzie"],
    "free_text": "gedichten over identiteit en opgroeien",
    "level": "2F",
    "avoid": [],
    "relevant": codes(160, 161, 252),
}

fixed_readers = [daan, thomas, fatima, milan, lotte, youssef, emma]      # liggen vast: niet aanpassen

# Jullie eigen drie testlezers. Zelfde velden als hierboven; "relevant" is een lijst cataloguscodes.
# Voorbeeld:
# reader_8 = {
#     "name": "...",
#     "tags": [...],
#     "genres": [...],
#     "free_text": "...",
#     "level": "2F",              # 2F, 3F of 3F+
#     "avoid": [],                # types materiaal die deze lezer niet wil
#     "relevant": codes(...),     # titels die deze lezer zou moeten krijgen
# }
# Titel en omschrijving bij een lijst codes: catalog.loc[codes(1, 17), ["Titel", "Niveau", "Korte omschrijving"]]
own_readers = []

readers = fixed_readers + own_readers

pd.DataFrame({"name": [r["name"] for r in readers],
              "level": [r["level"] for r in readers],
              "ingevuld": [", ".join(f for f in ["tags", "genres", "free_text"] if r[f]) for r in readers],
              "aantal relevant": [len(r["relevant"]) for r in readers]})

## 2 · Eén lijst meten: precision, recall en hit

Je hebt nu twee lijsten met codes: `recommended`, de lijst van de recommender met de beste titel vooraan, en `relevant`, de verwachting. Een lezer ziet alleen de eerste `k` titels, dus je meet op de top k. Een *treffer* is een titel uit de top k die ook in `relevant` staat.

| Maat | Vraag | Berekening |
|---|---|---|
| **precision@k** | Welk deel van wat de lezer ziet, is goed? | treffers in de top k, gedeeld door k |
| **recall@k** | Welk deel van wat goed is, krijgt de lezer te zien? | treffers in de top k, gedeeld door het aantal relevante titels |
| **hit@k** | Zit er minstens één goede titel bij? | 1 als er een treffer in de top k staat, anders 0 |

De maten zeggen niet hetzelfde. Voor Emma passen er maar drie titels: bij k = 5 komt haar precision nooit boven 0,6. Voor Daan passen er bijna dertig: bij k = 5 komt zijn recall nooit boven 0,17. Vergelijk een maat daarom tussen recommenders voor dezelfde lezers, niet tussen lezers. Kies `k` bewust: het is het aantal titels dat de lezer in jullie app te zien krijgt.

`precision_at_k` staat hieronder uitgewerkt als voorbeeld. `recall_at_k` (in pseudocode) en `hit_at_k` (in stappen) schrijf je zelf; de cel na de opdracht is de test.

In [ ]:
def precision_at_k(recommended, relevant, k):
    """Share of the top-k recommended titles that are relevant."""
    hits = sum(1 for t in recommended[:k] if t in set(relevant))
    return hits / k

def recall_at_k(recommended, relevant, k):
    """Share of the relevant titles that appear in the top-k."""
    # als relevant leeg is: geef 0.0 terug (delen door nul kan niet)
    # top  = de eerste k titels van recommended
    # hits = het aantal titels in top dat ook in relevant staat
    # geef terug: hits gedeeld door het aantal relevante titels
    raise NotImplementedError

def hit_at_k(recommended, relevant, k):
    """1 when at least one relevant title is in the top-k, otherwise 0."""
    # - Kijk alleen naar de eerste k titels.
    # - Staat er minstens een relevante titel bij: geef 1 terug, anders 0.
    raise NotImplementedError

# voorbeeld: twee van de eerste drie titels zijn een treffer voor Emma
recommended = codes(160, 21, 161, 50, 252)
print("precision@3 voor Emma:", round(precision_at_k(recommended, emma["relevant"], k=3), 2))

> **Opdracht 1 · recall_at_k en hit_at_k.** Werk beide functies uit; `precision_at_k` is je voorbeeld. Reken daarna met de hand na wat precision@5, recall@5 en hit@5 zijn voor de voorbeeldlijst hierboven en Emma, en controleer of je functies hetzelfde geven.

In [ ]:
# test: letters in plaats van codes, zodat je het met de hand kunt narekenen
rec = ["a", "b", "c", "d", "e"]
rel = ["b", "e", "z"]

assert precision_at_k(rec, rel, k=3) == 1 / 3
assert recall_at_k(rec, rel, k=3) == 1 / 3          # 1 van de 3 relevante titels
assert recall_at_k(rec, rel, k=5) == 2 / 3          # "z" wordt nooit aanbevolen
assert recall_at_k(rec, [], k=5) == 0.0             # geen verwachting: geen recall
assert hit_at_k(rec, rel, k=1) == 0                 # "a" is geen treffer
assert hit_at_k(rec, rel, k=2) == 1
assert hit_at_k([], rel, k=5) == 0                  # lege lijst: geen treffer
print("recall_at_k en hit_at_k kloppen")

## 3 · Alle testlezers, en een ondergrens

Eén lezer zegt weinig: de lijst kan voor Daan goed zijn en voor de rest niet. Je draait de meting daarom voor alle testlezers en kijkt naar de tabel per lezer én naar het gemiddelde.

Daarvoor spreek je één vorm af. Een **recommender** is in dit notebook een functie die een lezer (dict) krijgt en een lijst cataloguscodes teruggeeft, de beste vooraan. Hoe die lijst tot stand komt, maakt voor de meting niet uit.

Een cijfer betekent pas iets als je weet waarmee je het vergelijkt. Daarvoor heb je een **ondergrens** nodig: een simpele recommender zonder vectoren, waar jullie systeem overheen moet komen. Kies één van deze drie:

| Ondergrens | Hoe werkt het | Wat zegt het |
|---|---|---|
| **Willekeurig** | Geef n willekeurige titels terug. | Dit is de ondergrens: alles wat je bouwt moet hier ruim boven zitten. |
| **Populairst** | Geef iedereen dezelfde lijst met de meest geleende of best beoordeelde titels. | Dit kan alleen als jullie catalogus zo'n kolom heeft, en het is berucht moeilijk te verslaan. |
| **Zelfde genre** | Kies willekeurige titels uit de genres die de lezer aankruiste. | Dit is persoonlijk zonder vectoren, en laat zien wat de omschrijving en trefwoorden nog toevoegen bovenop het genre. |

`baseline_recommender` en `evaluate` schrijf je zelf.

In [ ]:
def baseline_recommender(person, n=10):
    """Baseline: a simple recommender without vectors. Returns a list of n catalogue codes."""
    # Kies een van de drie en werk hem uit.
    #
    # Willekeurig
    #   - trek n willekeurige codes uit catalog.index (random.sample)
    #   - zorg dat dezelfde lezer steeds dezelfde lijst krijgt, bijvoorbeeld met random.Random(person["name"])
    #
    # Populairst
    #   - sorteer de catalogus op een kolom die iets zegt over populariteit, de hoogste eerst
    #   - geef de eerste n codes terug; iedereen krijgt dezelfde lijst
    #
    # Zelfde genre
    #   - kies de titels waarvan het genre in person["genres"] staat
    #   - trek daaruit n willekeurige codes
    #   - wat doe je met een lezer die geen genre aankruiste, of een genre met minder dan n titels?
    raise NotImplementedError

def evaluate(recommender, readers, k=5):
    """One row per reader with precision, recall and hit at k."""
    # - Loop over de testlezers.
    # - Geef de recommender een kopie van de lezer zonder het veld "relevant": de verwachting mag
    #   nooit bij de recommender terechtkomen.
    # - Bereken precision_at_k, recall_at_k en hit_at_k voor de lijst die terugkomt.
    # - Geef een DataFrame terug met de kolommen "name", "precision", "recall" en "hit",
    #   een rij per lezer, in de volgorde van readers.
    raise NotImplementedError

> **Opdracht 2 · evaluate, een ondergrens en jullie eigen recommender.** Schrijf `evaluate`; de cel hieronder is de test. Kies daarna één van de drie ondergrenzen en werk hem uit in `baseline_recommender`. Koppel tot slot jullie eigen recommender (de laatste cel van dit deel) en leg de twee uitkomsten naast elkaar. Noteer in het keuzelog welke ondergrens je koos en waarom, het gemiddelde per maat voor de ondergrens en voor jullie recommender, en voor welke lezer jullie recommender het slechtst scoort. Ligt dat aan de recommender of aan de catalogus?

In [ ]:
# test: een recommender met een vaste lijst voor Emma en Lotte, met de hand na te rekenen
FIXED = {
    "Emma":  codes(160, 1, 161, 2, 3),       # 160 en 161 zijn relevant voor Emma
    "Lotte": codes(5, 21, 6, 7, 8),          # alleen 21 is relevant voor Lotte
}

def fixed_recommender(person):
    assert "relevant" not in person, "de recommender krijgt het veld relevant te zien"
    return FIXED[person["name"]]

result = evaluate(fixed_recommender, [emma, lotte], k=3)
assert list(result.columns) == ["name", "precision", "recall", "hit"], list(result.columns)
assert list(result["name"]) == ["Emma", "Lotte"]

by_name = result.set_index("name")
assert abs(by_name.loc["Emma", "precision"] - 2 / 3) < 1e-9                              # 2 treffers in de top 3
assert abs(by_name.loc["Emma", "recall"] - 2 / len(emma["relevant"])) < 1e-9
assert abs(by_name.loc["Lotte", "precision"] - 1 / 3) < 1e-9
assert abs(by_name.loc["Lotte", "recall"] - 1 / len(lotte["relevant"])) < 1e-9
assert list(result["hit"]) == [1, 1]
assert list(evaluate(fixed_recommender, [emma, lotte], k=1)["hit"]) == [1, 0]

print("evaluate klopt")
result

### Jullie eigen recommender meten

`recommend_for` uit de vorige les geeft een DataFrame terug; de evaluatie verwacht een lijst cataloguscodes. Daar zit één regel tussen. De cel hieronder is een startpunt: haal de `#` weg en pas de namen aan jullie eigen code aan.

Per lezer is hit een 0 of een 1. De gemiddelde hit is dus het deel van de lezers met minstens één treffer: 0,3 betekent 3 van de 10 lezers.

In [ ]:
# from recommender import recommend_for
#
# def my_recommender(person):
#     """Our own recommender in the shape the evaluation expects: a list of codes, best first."""
#     return list(recommend_for(person, n=10)["cataloguscode"])
#
# baseline = evaluate(baseline_recommender, readers, k=5)
# ours     = evaluate(my_recommender, readers, k=5)
#
# print("ondergrens:", baseline[["precision", "recall", "hit"]].mean().round(2).to_dict())
# print("wij:       ", ours[["precision", "recall", "hit"]].mean().round(2).to_dict())
# ours.round(2)                                        # per lezer: wie gaat goed, wie niet?

---
## Opdracht

**Doel.** Jullie kunnen met cijfers laten zien hoe goed jullie recommender is, vergeleken met een simpele ondergrens.

*Klaar als.* `evaluate` geeft voor de tien testlezers een tabel voor de ondergrens en een voor jullie eigen recommender, en het keuzelog legt uit wat het verschil is.

**Extra**
- [ ] Een maat die ook naar de volgorde kijkt: een treffer op plek 1 telt zwaarder dan een treffer op plek 5.
- [ ] Een testlezer die niets invulde: wat is voor de koude start een goede verwachting?